# Selenium HTML Field Mapper

This notebook parses through the AllSci application HTML to map all data fields and create a comprehensive field mapping showing:
- Page URL
- Field label/name
- Field value
- CSS selector
- Data type/category

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urlparse, parse_qs
import json
from datetime import datetime

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Login credentials
SUPABASE_EMAIL = "rlalani@allsci.com"
SUPABASE_PASSWORD = "!!Casio1994$$"

# Application URLs to scan
URLS_TO_SCAN = {
    "Production": {
        "login": "https://app.allsci.com/?login=true",
        "pages": [
            "https://app.allsci.com/explore/clinical-trials",
            # Add more pages to scan here
        ]
    },
    # "Staging": {
    #     "login": "https://app-stg.allsci.com/?login=true",
    #     "pages": [
    #         "https://app-stg.allsci.com/explore/clinical-trials",
    #     ]
    # }
}

# Wait times
PAGE_LOAD_WAIT = 30  # seconds to wait for page load
ELEMENT_WAIT = 10    # seconds to wait for specific elements

# Field detection patterns
FIELD_PATTERNS = {
    'metadata': {
        'selector': 'div#metadata-content h1',
        'pattern': r'^([^:]+):\s*(.+)$'
    },
    'buttons_with_data': {
        'selector': 'button[aria-label]',
        'pattern': None
    },
    'labeled_spans': {
        'selector': 'span.flex.flex-row.items-center',
        'pattern': None
    },
    'data_attributes': {
        'selector': '[data-testid], [data-field], [data-value]',
        'pattern': None
    }
}

## Field Extraction Functions

In [ ]:
def setup_driver(headless=False):
    """Setup Chrome WebDriver with appropriate options."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-gpu')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def login_to_application(driver, login_url, email, password):
    """Login to the application using Supabase credentials."""
    print(f"Navigating to login: {login_url}")
    driver.get(login_url)
    
    try:
        # Wait for email input
        email_input = WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.NAME, "email"))
        )
        email_input.send_keys(email)
        print("  Email entered")
        
        # Enter password
        password_input = driver.find_element(By.NAME, "password")
        password_input.send_keys(password)
        print("  Password entered")
        
        # Click Sign In button
        sign_in_button = driver.find_element(By.XPATH, "//button[@type='submit' and contains(text(), 'Sign In')]")
        sign_in_button.click()
        print("  Sign In clicked")
        
        # Wait for navigation away from login page
        time.sleep(5)
        print("  Login successful!")
        return True
        
    except Exception as e:
        print(f"  Login failed: {e}")
        return False


def get_css_selector(element, driver):
    """Generate a CSS selector for an element."""
    try:
        # Try to get a unique selector using JavaScript
        selector = driver.execute_script("""
            function getCssPath(el) {
                if (!(el instanceof Element)) return;
                var path = [];
                while (el.nodeType === Node.ELEMENT_NODE) {
                    var selector = el.nodeName.toLowerCase();
                    if (el.id) {
                        selector += '#' + el.id;
                        path.unshift(selector);
                        break;
                    } else {
                        var sib = el, nth = 1;
                        while (sib = sib.previousElementSibling) {
                            if (sib.nodeName.toLowerCase() == selector)
                                nth++;
                        }
                        if (nth != 1)
                            selector += ":nth-of-type("+nth+")";
                    }
                    path.unshift(selector);
                    el = el.parentNode;
                }
                return path.join(" > ");
            }
            return getCssPath(arguments[0]);
        """, element)
        return selector
    except:
        return "Unknown"


def extract_metadata_fields(driver):
    """Extract metadata fields like Conditions, Intervention, Study Type, Phase, etc."""
    fields = []
    
    try:
        # Wait for metadata section to load
        WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.ID, "metadata-content"))
        )
        
        # Find all metadata field elements
        metadata_elements = driver.find_elements(By.CSS_SELECTOR, "div#metadata-content h1")
        
        for element in metadata_elements:
            text = element.text.strip()
            if ':' in text:
                label, value = text.split(':', 1)
                label = label.strip()
                value = value.strip()
                
                # Try to find the span containing the value
                try:
                    value_span = element.find_element(By.TAG_NAME, "span")
                    value = value_span.text.strip()
                except:
                    pass
                
                fields.append({
                    'category': 'metadata',
                    'label': label,
                    'value': value,
                    'selector': get_css_selector(element, driver),
                    'element_type': 'h1',
                    'has_data': bool(value)
                })
    except Exception as e:
        print(f"  Warning: Could not extract metadata fields: {e}")
    
    return fields


def extract_button_metrics(driver):
    """Extract metrics from buttons like citation counts, etc."""
    fields = []
    
    try:
        # Find all buttons with aria-label (these often contain metrics)
        buttons = driver.find_elements(By.CSS_SELECTOR, "button[aria-label]")
        
        for button in buttons:
            aria_label = button.get_attribute('aria-label')
            text = button.text.strip()
            
            # Check if button contains numerical data
            if text and (text.isdigit() or re.search(r'\d+', text)):
                fields.append({
                    'category': 'button_metric',
                    'label': aria_label,
                    'value': text,
                    'selector': get_css_selector(button, driver),
                    'element_type': 'button',
                    'has_data': True
                })
    except Exception as e:
        print(f"  Warning: Could not extract button metrics: {e}")
    
    return fields


def extract_date_fields(driver):
    """Extract date fields from the page."""
    fields = []
    
    try:
        # Look for spans containing calendar icons and dates
        date_elements = driver.find_elements(By.XPATH, 
            "//span[contains(@class, 'flex') and .//svg and text()]")
        
        for element in date_elements:
            text = element.text.strip()
            # Check if it looks like a year or date
            if re.match(r'^\d{4}$', text) or re.match(r'\d{1,2}/\d{1,2}/\d{2,4}', text):
                fields.append({
                    'category': 'date',
                    'label': 'Date/Year',
                    'value': text,
                    'selector': get_css_selector(element, driver),
                    'element_type': 'span',
                    'has_data': True
                })
    except Exception as e:
        print(f"  Warning: Could not extract date fields: {e}")
    
    return fields


def extract_all_text_fields(driver):
    """Extract all text fields with labels (fallback method)."""
    fields = []
    
    try:
        # Use BeautifulSoup for additional parsing
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find all elements with data attributes
        data_elements = soup.find_all(attrs={'data-testid': True})
        for elem in data_elements:
            testid = elem.get('data-testid')
            text = elem.get_text(strip=True)
            if text:
                fields.append({
                    'category': 'data_testid',
                    'label': testid,
                    'value': text[:100],  # Limit to 100 chars
                    'selector': f'[data-testid="{testid}"]',
                    'element_type': elem.name,
                    'has_data': True
                })
    except Exception as e:
        print(f"  Warning: Could not extract text fields: {e}")
    
    return fields


def extract_table_data(driver):
    """Extract data from tables if present."""
    fields = []
    
    try:
        tables = driver.find_elements(By.TAG_NAME, "table")
        
        for idx, table in enumerate(tables):
            # Get table headers
            headers = []
            try:
                header_cells = table.find_elements(By.TAG_NAME, "th")
                headers = [h.text.strip() for h in header_cells]
            except:
                pass
            
            # Get first data row as sample
            try:
                rows = table.find_elements(By.TAG_NAME, "tr")
                if len(rows) > 1:
                    cells = rows[1].find_elements(By.TAG_NAME, "td")
                    for cell_idx, cell in enumerate(cells):
                        label = headers[cell_idx] if cell_idx < len(headers) else f"Column {cell_idx + 1}"
                        value = cell.text.strip()
                        
                        fields.append({
                            'category': 'table',
                            'label': f"Table {idx + 1} - {label}",
                            'value': value[:100],
                            'selector': get_css_selector(cell, driver),
                            'element_type': 'td',
                            'has_data': bool(value)
                        })
            except:
                pass
    except Exception as e:
        print(f"  Warning: Could not extract table data: {e}")
    
    return fields


def extract_all_fields_from_page(driver, page_url):
    """Extract all fields from the current page."""
    print(f"\n  Extracting fields from: {page_url}")
    
    all_fields = []
    
    # Extract different types of fields
    print("    - Extracting metadata fields...")
    all_fields.extend(extract_metadata_fields(driver))
    
    print("    - Extracting button metrics...")
    all_fields.extend(extract_button_metrics(driver))
    
    print("    - Extracting date fields...")
    all_fields.extend(extract_date_fields(driver))
    
    print("    - Extracting table data...")
    all_fields.extend(extract_table_data(driver))
    
    print("    - Extracting data-testid fields...")
    all_fields.extend(extract_all_text_fields(driver))
    
    # Add page URL to each field
    for field in all_fields:
        field['page_url'] = page_url
        field['page_title'] = driver.title
    
    print(f"    Total fields extracted: {len(all_fields)}")
    return all_fields

## Click-through and Deep Scan Functions

In [ ]:
def wait_for_page_load(driver, timeout=PAGE_LOAD_WAIT):
    """Wait for page to finish loading."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )
        # Additional wait for any async content
        time.sleep(2)
    except TimeoutException:
        print("    Warning: Page load timeout")


def click_first_trial_and_extract(driver, base_url):
    """Click on the first trial in the atlas and extract detail page fields."""
    fields = []
    
    try:
        print("\n  Attempting to click first trial...")
        
        # Wait a bit for the atlas to load
        time.sleep(5)
        
        # Try to find clickable trial elements
        # This might need adjustment based on actual HTML structure
        trial_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/trial/'], a[href*='/clinical-trial/']")
        
        if trial_links:
            first_trial = trial_links[0]
            trial_url = first_trial.get_attribute('href')
            print(f"    Clicking trial: {trial_url}")
            first_trial.click()
            
            # Wait for detail page to load
            wait_for_page_load(driver)
            
            # Extract fields from detail page
            detail_fields = extract_all_fields_from_page(driver, driver.current_url)
            fields.extend(detail_fields)
        else:
            print("    No trial links found to click")
            
    except Exception as e:
        print(f"    Warning: Could not click trial: {e}")
    
    return fields

## Main Scanning Function

In [ ]:
def scan_application(urls_config, headless=False):
    """Scan the application and extract all field mappings."""
    
    driver = setup_driver(headless=headless)
    all_fields = []
    
    try:
        for env_name, config in urls_config.items():
            print(f"\n{'='*60}")
            print(f"Scanning Environment: {env_name}")
            print(f"{'='*60}")
            
            # Login
            login_success = login_to_application(
                driver, 
                config['login'], 
                SUPABASE_EMAIL, 
                SUPABASE_PASSWORD
            )
            
            if not login_success:
                print(f"  Skipping {env_name} due to login failure")
                continue
            
            # Scan each page
            for page_url in config['pages']:
                print(f"\n{'='*60}")
                print(f"Scanning Page: {page_url}")
                print(f"{'='*60}")
                
                # Navigate to page
                driver.get(page_url)
                wait_for_page_load(driver)
                
                # Extract fields from main page
                page_fields = extract_all_fields_from_page(driver, page_url)
                
                # Add environment info
                for field in page_fields:
                    field['environment'] = env_name
                    field['scan_timestamp'] = datetime.now().isoformat()
                
                all_fields.extend(page_fields)
                
                # Try to click into a detail page and extract more fields
                detail_fields = click_first_trial_and_extract(driver, page_url)
                for field in detail_fields:
                    field['environment'] = env_name
                    field['scan_timestamp'] = datetime.now().isoformat()
                    field['is_detail_page'] = True
                
                all_fields.extend(detail_fields)
    
    finally:
        driver.quit()
    
    return all_fields

## Run the Scan

In [ ]:
# Run the scan
print("Starting application field mapping scan...\n")
fields = scan_application(URLS_TO_SCAN, headless=False)

print(f"\n{'='*60}")
print(f"Scan Complete! Total fields extracted: {len(fields)}")
print(f"{'='*60}")

## Create Field Mapping DataFrame

In [ ]:
# Convert to DataFrame
df_fields = pd.DataFrame(fields)

# Display summary
print("\nField Mapping Summary:")
print(f"Total fields: {len(df_fields)}")
print(f"\nFields by category:")
print(df_fields['category'].value_counts())
print(f"\nFields with data: {df_fields['has_data'].sum()}")
print(f"Fields without data: {(~df_fields['has_data']).sum()}")

# Show sample of fields
print("\nSample of extracted fields:")
df_fields.head(20)

## Analyze Field Coverage by Page

In [ ]:
# Group by page and category
page_summary = df_fields.groupby(['page_url', 'category']).agg({
    'label': 'count',
    'has_data': 'sum'
}).rename(columns={'label': 'total_fields', 'has_data': 'fields_with_data'})

print("\nField Coverage by Page and Category:")
print(page_summary)

## Export Field Mapping

In [ ]:
# Export to CSV
output_file = f'field_mapping_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_fields.to_csv(output_file, index=False)
print(f"\nField mapping exported to: {output_file}")

# Export to JSON for more structured data
json_file = output_file.replace('.csv', '.json')
df_fields.to_json(json_file, orient='records', indent=2)
print(f"Field mapping also exported to: {json_file}")

## Detailed Field Analysis

In [ ]:
# Show all unique field labels by category
print("\nUnique Fields by Category:\n")
for category in df_fields['category'].unique():
    print(f"\n{category.upper()}:")
    category_fields = df_fields[df_fields['category'] == category]
    unique_labels = category_fields['label'].unique()
    for label in unique_labels[:20]:  # Show first 20
        sample_value = category_fields[category_fields['label'] == label]['value'].iloc[0]
        print(f"  - {label}: {sample_value[:50]}..." if len(str(sample_value)) > 50 else f"  - {label}: {sample_value}")
    if len(unique_labels) > 20:
        print(f"  ... and {len(unique_labels) - 20} more")

## Field Validation Report

In [ ]:
# Create a validation report
validation_report = []

for _, field in df_fields.iterrows():
    validation_report.append({
        'Page': field['page_url'].split('/')[-1] or 'home',
        'Category': field['category'],
        'Field Label': field['label'],
        'Has Data': 'Yes' if field['has_data'] else 'No',
        'Sample Value': str(field['value'])[:50],
        'CSS Selector': field['selector'][:80] if len(str(field['selector'])) > 80 else field['selector']
    })

df_validation = pd.DataFrame(validation_report)

print("\nField Validation Report:")
print(df_validation.to_string(max_rows=50))

# Export validation report
validation_file = f'validation_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_validation.to_csv(validation_file, index=False)
print(f"\nValidation report exported to: {validation_file}")